In [5]:
import random
import time
from typing import Literal, Optional, TypedDict
from langgraph.graph import END, START, StateGraph


# ==========================================
# 1. Day 24 扩展后的 AgentState
# ==========================================
class AgentState(TypedDict):
    employee_id: str
    amount: float

    # API Attempt 状态
    api_status: Optional[int]
    error_msg: Optional[str]

    # Retry 策略状态
    retry_count: int
    max_retries: int
    retry_delay: float

    # Deadline 策略状态
    deadline: float
    request_timeout: float

    # External Interruption 状态
    cancelled: bool

    # Policy 决策输出
    policy_action: Optional[
        Literal[
            "ALLOW",
            "FAST_FAIL",
            "SUCCESS",
            "RETRY",
            "FALLBACK",
            "DEADLINE_EXCEEDED",
            "CANCELLED",
            "FATAL_ERROR",
        ]
    ]

    # 执行结果
    result: Optional[str]


# ==========================================
# 2. 共享 Circuit Breaker 实例 (Runtime Shared)
# ==========================================
class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, cooldown: float = 5.0):
        self.state: str = "CLOSED"  # "CLOSED", "OPEN", "HALF_OPEN"
        self.failure_count: int = 0
        self.failure_threshold: int = failure_threshold
        self.cooldown: float = cooldown
        self.opened_at: Optional[float] = None

    def can_call(self) -> bool:
        now = time.time()
        if self.state == "OPEN":
            if self.opened_at and (now - self.opened_at >= self.cooldown):
                self.state = "HALF_OPEN"
                return True
            return False
        return True

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        if self.state == "HALF_OPEN" or self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            self.opened_at = time.time()


# 全局共享 Breaker 实例
hr_api_breaker = CircuitBreaker()


# ==========================================
# 3. 5 大核心 Node 实现
# ==========================================
def circuit_breaker_gate_node(state: AgentState) -> dict:
    """1. 入口门控：检查共享 Breaker 状态"""
    if hr_api_breaker.can_call():
        return {"policy_action": "ALLOW"}
    return {"policy_action": "FAST_FAIL"}


def call_hr_api_node(state: AgentState) -> dict:
    """2. 单次 API 尝试 (不做 sleep 或循环)"""
    status = state.get("api_status", 503)
    error_msg = "OK" if status == 200 else f"HTTP Error {status}"
    return {"api_status": status, "error_msg": error_msg}


def policy_node(state: AgentState) -> dict:
    """3. 策略决策中心 (Cancellation, Success, Fatal, Half-Open Probe, Retry, Deadline)"""
    # 1. 外部取消优先判定
    if state.get("cancelled", False):
        return {"policy_action": "CANCELLED"}

    status = state.get("api_status")

    # 2. 成功判定
    if status == 200:
        hr_api_breaker.record_success()
        return {"policy_action": "SUCCESS", "result": "API Call Succeeded"}

    # 失败上报给共享 Breaker
    hr_api_breaker.record_failure()

    # 3. 不可重试致命错误判定
    if status in (400, 401, 403, 404, 422, 501):
        return {"policy_action": "FATAL_ERROR", "result": f"Fatal Error: {status}"}

    # HALF_OPEN 探测失败判定：直接降级，避免二次倾泻流量
    if hr_api_breaker.state == "OPEN":
        return {"policy_action": "FALLBACK", "result": "Half-Open Probe Failed -> Fallback"}

    # 4. 重试次数上限检查
    if state["retry_count"] >= state["max_retries"]:
        return {"policy_action": "FALLBACK", "result": "Max Retries Reached -> Fallback"}

    # 5. Backoff + Jitter 计算
    backoff = state.get("retry_delay", 1.0) * (2 ** state["retry_count"])
    jitter = random.uniform(0, 0.5)
    computed_delay = backoff + jitter

    # 6. Deadline 预算检查
    now = time.time()
    remaining_budget = state["deadline"] - now
    required_budget = computed_delay + state["request_timeout"] + 0.1

    if remaining_budget < required_budget:
        return {"policy_action": "DEADLINE_EXCEEDED", "result": "Deadline Exceeded -> Fallback"}

    return {"policy_action": "RETRY", "retry_delay": computed_delay}


def retry_wait_node(state: AgentState) -> dict:
    """4. 执行退避等待"""
    time.sleep(state["retry_delay"])
    return {"retry_count": state["retry_count"] + 1}


def fallback_node(state: AgentState) -> dict:
    """5. 降级执行节点"""
    return {"result": f"Fallback Executed. Trigger Action: {state.get('policy_action')}"}


# ==========================================
# 4. Router 与 Graph 编排
# ==========================================
def route_gate(state: AgentState) -> str:
    if state["policy_action"] == "ALLOW":
        return "call_hr_api"
    return "fallback"


def route_policy(state: AgentState) -> str:
    action = state["policy_action"]
    if action == "SUCCESS":
        return "end"
    if action == "RETRY":
        return "retry_wait"
    if action in ("FALLBACK", "DEADLINE_EXCEEDED"):
        return "fallback"
    if action in ("CANCELLED", "FATAL_ERROR"):
        return "end"
    return "end"


builder = StateGraph(AgentState)

builder.add_node("circuit_breaker_gate", circuit_breaker_gate_node)
builder.add_node("call_hr_api", call_hr_api_node)
builder.add_node("policy", policy_node)
builder.add_node("retry_wait", retry_wait_node)
builder.add_node("fallback", fallback_node)

builder.set_entry_point("circuit_breaker_gate")

builder.add_conditional_edges(
    "circuit_breaker_gate",
    route_gate,
    {"call_hr_api": "call_hr_api", "fallback": "fallback"},
)

builder.add_edge("call_hr_api", "policy")

builder.add_conditional_edges(
    "policy",
    route_policy,
    {
        "end": END,
        "retry_wait": "retry_wait",
        "fallback": "fallback",
    },
)

# 关键回环：retry_wait 直连 call_hr_api，跳过 Circuit Breaker Gate
builder.add_edge("retry_wait", "call_hr_api")
builder.add_edge("fallback", END)

graph = builder.compile()


# ==========================================
# 5. 验证执行脚本 (Scenarios)
# ==========================================
if __name__ == "__main__":
    print("--- Scenario 1: Retry Loop until Max Retries Exceeded ---")
    hr_api_breaker = CircuitBreaker()
    state_1: AgentState = {
        "employee_id": "EMP_001",
        "amount": 100.0,
        "api_status": 503,
        "error_msg": None,
        "retry_count": 0,
        "max_retries": 2,
        "retry_delay": 0.1,
        "deadline": time.time() + 10.0,
        "request_timeout": 1.0,
        "cancelled": False,
        "policy_action": None,
        "result": None,
    }
    res_1 = graph.invoke(state_1)
    print(f"Action: {res_1.get('policy_action')} | Retries: {res_1.get('retry_count')} | Result: {res_1.get('result')}\n")

    print("--- Scenario 2: Deadline Exceeded ---")
    hr_api_breaker = CircuitBreaker()
    state_2: AgentState = {
        "employee_id": "EMP_002",
        "amount": 200.0,
        "api_status": 500,
        "error_msg": None,
        "retry_count": 1,
        "max_retries": 5,
        "retry_delay": 2.0,
        "deadline": time.time() + 1.5,  # 时间预算不足
        "request_timeout": 1.0,
        "cancelled": False,
        "policy_action": None,
        "result": None,
    }
    res_2 = graph.invoke(state_2)
    print(f"Action: {res_2.get('policy_action')} | Result: {res_2.get('result')}\n")

    print("--- Scenario 3: External Cancellation ---")
    state_3: AgentState = {
        "employee_id": "EMP_003",
        "amount": 300.0,
        "api_status": 200,
        "error_msg": None,
        "retry_count": 0,
        "max_retries": 3,
        "retry_delay": 0.1,
        "deadline": time.time() + 10.0,
        "request_timeout": 1.0,
        "cancelled": True,  # 已取消
        "policy_action": None,
        "result": None,
    }
    res_3 = graph.invoke(state_3)
    print(f"Action: {res_3.get('policy_action')} | Result: {res_3.get('result')}")

--- Scenario 1: Retry Loop until Max Retries Exceeded ---
Action: FALLBACK | Retries: 2 | Result: Fallback Executed. Trigger Action: FALLBACK

--- Scenario 2: Deadline Exceeded ---
Action: DEADLINE_EXCEEDED | Result: Fallback Executed. Trigger Action: DEADLINE_EXCEEDED

--- Scenario 3: External Cancellation ---
Action: CANCELLED | Result: None
